# FIN_DB – Account Data Platform Setup (Raw Layer)

**Use case:** Daily financial-company account data lands in the Snowflake **RAW** layer as a full snapshot. dbt later loads it into the **SILVER** change-log table, inserting a row only when attributes change.

| Item | Value |
|---|---|
| Natural key | `SLSFRC_ID` + `EFECTIVE_DT` |
| Role | `FIN_DBT_DEV_ROLE` |
| Warehouse | `FIN_DBT_WH` (XSMALL) |
| Database / schemas | `FIN_DB.RAW`, `FIN_DB.SILVER` |

> These steps were already executed on 2026-11-05. Steps 1–3 need **ACCOUNTADMIN**; steps 4–7 run as **FIN_DBT_DEV_ROLE**. Every statement is idempotent (`IF NOT EXISTS` / `OR REPLACE`), so re-running is safe; Step 4 recreates the table, which resets the data before Step 5 reloads it.

## Step 1 – Developer role
Create the role dbt runs as, and grant it to SYSADMIN (to keep the role hierarchy) and to the developer user.

In [ ]:
%%sql -r step1_role
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS FIN_DBT_DEV_ROLE;
GRANT ROLE FIN_DBT_DEV_ROLE TO ROLE SYSADMIN;
GRANT ROLE FIN_DBT_DEV_ROLE TO USER SUSSAHU;

## Step 2 – Warehouse
An XSMALL warehouse that auto-suspends after 60 seconds idle, to keep cost low.

In [ ]:
%%sql -r step2_wh
CREATE WAREHOUSE IF NOT EXISTS FIN_DBT_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
GRANT USAGE, OPERATE ON WAREHOUSE FIN_DBT_WH TO ROLE FIN_DBT_DEV_ROLE;

## Step 3 – Database, schemas and grants
- `RAW`: as-landed daily snapshots
- `SILVER`: curated change log, built by dbt
- `EXECUTE TASK`: lets the role schedule the dbt run

In [ ]:
%%sql -r step3_db
CREATE DATABASE IF NOT EXISTS FIN_DB;
CREATE SCHEMA IF NOT EXISTS FIN_DB.RAW;
CREATE SCHEMA IF NOT EXISTS FIN_DB.SILVER;

GRANT USAGE ON DATABASE FIN_DB TO ROLE FIN_DBT_DEV_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA FIN_DB.RAW    TO ROLE FIN_DBT_DEV_ROLE;
GRANT ALL PRIVILEGES ON SCHEMA FIN_DB.SILVER TO ROLE FIN_DBT_DEV_ROLE;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE FIN_DBT_DEV_ROLE;

## Step 4 – Raw ACCOUNT table
Switch to the developer role so it owns the raw objects. `INGESTED_TS` is the load timestamp that dbt uses for incremental CDC.

In [ ]:
%%sql -r step4_table
USE ROLE FIN_DBT_DEV_ROLE;
USE WAREHOUSE FIN_DBT_WH;

CREATE OR REPLACE TABLE FIN_DB.RAW.ACCOUNT (
    SLSFRC_ID       VARCHAR(18),
    EFECTIVE_DT     DATE,
    ACCOUNT_NAME    VARCHAR,
    ACCOUNT_TYPE    VARCHAR,
    INDUSTRY        VARCHAR,
    ANNUAL_REVENUE  NUMBER(18,2),
    ACCOUNT_STATUS  VARCHAR,
    BILLING_STATE   VARCHAR(2),
    INGESTED_TS     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## Step 5 – Synthetic daily snapshots (days 1–4)
| Day | EFECTIVE_DT | Scenario |
|---|---|---|
| 1 | 2026-11-01 | Initial snapshot, 5 accounts |
| 2 | 2026-11-02 | 001 revenue 5.0M → 5.5M, 002 Prospect → Customer, 006 new account |
| 3 | 2026-11-03 | No changes (dbt should insert nothing) |
| 4 | 2026-11-04 | 001 renamed **Acme CapitalOne**, 005 Inactive → Active |

In [ ]:
%%sql -r step5_day1_2
INSERT INTO FIN_DB.RAW.ACCOUNT VALUES
('001A000001','2026-11-01','Acme Capital',     'Customer','Banking',   5000000.00,'Active',  'NY','2026-11-01 06:00:00'),
('001A000002','2026-11-01','Blue River Wealth','Prospect','Asset Mgmt',1200000.00,'Active',  'CA','2026-11-01 06:00:00'),
('001A000003','2026-11-01','Cedar Insurance',  'Customer','Insurance', 8700000.00,'Active',  'IA','2026-11-01 06:00:00'),
('001A000004','2026-11-01','Delta Retirement', 'Partner', 'Pension',   3300000.00,'Active',  'TX','2026-11-01 06:00:00'),
('001A000005','2026-11-01','Evergreen Trust',  'Customer','Trust',      950000.00,'Inactive','WA','2026-11-01 06:00:00'),
('001A000001','2026-11-02','Acme Capital',     'Customer','Banking',   5500000.00,'Active',  'NY','2026-11-02 06:00:00'),
('001A000002','2026-11-02','Blue River Wealth','Customer','Asset Mgmt',1200000.00,'Active',  'CA','2026-11-02 06:00:00'),
('001A000003','2026-11-02','Cedar Insurance',  'Customer','Insurance', 8700000.00,'Active',  'IA','2026-11-02 06:00:00'),
('001A000004','2026-11-02','Delta Retirement', 'Partner', 'Pension',   3300000.00,'Active',  'TX','2026-11-02 06:00:00'),
('001A000005','2026-11-02','Evergreen Trust',  'Customer','Trust',      950000.00,'Inactive','WA','2026-11-02 06:00:00'),
('001A000006','2026-11-02','Falcon Advisors',  'Prospect','Advisory',   400000.00,'Active',  'IL','2026-11-02 06:00:00');

In [ ]:
%%sql -r step5_day3
INSERT INTO FIN_DB.RAW.ACCOUNT
SELECT SLSFRC_ID, '2026-11-03', ACCOUNT_NAME, ACCOUNT_TYPE, INDUSTRY, ANNUAL_REVENUE,
       ACCOUNT_STATUS, BILLING_STATE, '2026-11-03 06:00:00'
FROM FIN_DB.RAW.ACCOUNT WHERE EFECTIVE_DT = '2026-11-02';

In [ ]:
%%sql -r step5_day4
INSERT INTO FIN_DB.RAW.ACCOUNT
SELECT SLSFRC_ID, '2026-11-04',
       IFF(SLSFRC_ID = '001A000001', 'Acme CapitalOne', ACCOUNT_NAME),
       ACCOUNT_TYPE, INDUSTRY, ANNUAL_REVENUE,
       IFF(SLSFRC_ID = '001A000005', 'Active', ACCOUNT_STATUS),
       BILLING_STATE, '2026-11-04 06:00:00'
FROM FIN_DB.RAW.ACCOUNT WHERE EFECTIVE_DT = '2026-11-03';

## Step 6 – Source view
dbt reads from this view (`source('raw','vw_account_src')`), not from the table directly.

In [ ]:
%%sql -r step6_view
CREATE OR REPLACE VIEW FIN_DB.RAW.VW_ACCOUNT_SRC AS
SELECT SLSFRC_ID, EFECTIVE_DT, ACCOUNT_NAME, ACCOUNT_TYPE, INDUSTRY,
       ANNUAL_REVENUE, ACCOUNT_STATUS, BILLING_STATE, INGESTED_TS
FROM FIN_DB.RAW.ACCOUNT;

## Step 7 – Validation
Expected row counts: **5 / 6 / 6 / 6 = 23 raw rows**. The second query shows each account's attributes by day, so the changes dbt should detect are easy to see.

In [ ]:
%%sql -r raw_row_counts
SELECT EFECTIVE_DT, COUNT(*) AS ROWS_LOADED
FROM FIN_DB.RAW.VW_ACCOUNT_SRC
GROUP BY 1 ORDER BY 1;

In [ ]:
%%sql -r raw_detail
SELECT SLSFRC_ID, EFECTIVE_DT, ACCOUNT_NAME, ACCOUNT_TYPE, ANNUAL_REVENUE, ACCOUNT_STATUS, INGESTED_TS
FROM FIN_DB.RAW.VW_ACCOUNT_SRC
ORDER BY SLSFRC_ID, EFECTIVE_DT;

## Next steps – dbt project
1. Create the `fin_account_dbt` dbt project in Workspaces (`dbt_project.yml`, `dbt_projects_profiles.yml`, `sources.yml`, `stg_account.sql`, `slv_account.sql`, tests).
2. `dbt build --select slv_account --full-refresh` → expect **10 rows** in `FIN_DB.SILVER.SLV_ACCOUNT`, none dated 2026-11-03.
3. Deploy with `CREATE DBT PROJECT FIN_DB.SILVER.FIN_ACCOUNT_DBT`, then schedule it with a Task.